## dataset

In [1]:
!pip install tqdm fastparquet

In [179]:
import requests
import yaml
import getpass
# import zstandard as zstd
import pandas as pd
import json
import io

from typing import Dict, List, Any
from tqdm import tqdm

In [180]:
df_queries = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_queries = df_queries[df_queries["product_locale"] == "us"]


In [181]:
df_queries.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [182]:
unique_queries = df_queries["query"].drop_duplicates()

In [183]:
len(unique_queries)

97345

In [184]:
unique_queries.head(5)

0                                revent 80 cfm
16                !awnmower tires without rims
32                !qscreen fence without holes
149    # 10 self-seal envelopes without window
189                  # 2 pencils not sharpened
Name: query, dtype: object

In [185]:
random_queries = unique_queries.sample(n=1000, random_state=42)
len(random_queries)

1000

In [186]:
df_products = pd.read_parquet('esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_products = df_products[df_products["product_locale"] == "us"]
df_products.head(5)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
167168,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
167169,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
167170,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
167171,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
167172,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [187]:
df_random_queries = df_queries[
    df_queries["query"].isin(random_queries)
]

In [188]:
df = pd.merge(
    df_random_queries,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)
df.head(5)

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
0,3042,$5 items,100,B079HXXP4T,us,I,1,1,test,"Soft Scrub In-Tank Toilet Cleaner Duo-Cubes, A...",None,"Helps fight toilet ring, hard water, and limes...",Soft Scrub,Alpine Fresh
1,3043,$5 items,100,B07DZYGDS3,us,E,1,1,test,"Gillette Fusion5 Razors for Men, 1 Gillette Ra...",None,REFILLS FIT ALL GILLETTE 5-BLADE RAZOR HANDLES...,Gillette,None
2,3044,$5 items,100,B07HY9DC4N,us,E,1,1,test,"Summer's Eve Cleansing Cloths, Blissful Escape...",None,Summer's Eve Feminine Cleansing Wipes are safe...,Summer's Eve,None
3,3045,$5 items,100,B07M77RB97,us,E,1,1,test,BIC Flex 5 Hybrid Men's 5-Blade Disposable Raz...,None,"5 long lasting, flexible blades individually a...",BIC,Black
4,3046,$5 items,100,B07NTWYGJX,us,I,1,1,test,"6PCS Dual Heads Blackhead Remover, Pimple Come...",<b>About Our Factory:</b><br /> ✿✿Our factory ...,"♥ 【Dual Heads REMOVER】: 6PCS dual heads tools,...",USCOLOR,None


In [189]:
len(df)

18727

## index

In [190]:
SEARCH_INDEX = 'http://localhost:9200/megacities'

In [191]:
idx = requests.put(
    SEARCH_INDEX,
    json={
        "mappings": {
            "properties": {
                "name": {
                    "type": "text"
                },
                "description": {
                    "type": "text"
                }
            }
        }
    }
)
idx.json()

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'megacities'}

In [192]:
def index_record(id, name, description):
    if id and name and description:
        try:
            return requests.post(
                f"{SEARCH_INDEX}/_doc/{id}",
                json={
                    'name': name,
                    'description': description    
                }
            )
        except:
            pass

In [193]:
for index, row in tqdm(df.iterrows(), total=len(df)):
    _ = index_record(row['example_id'], row['product_title'], row['product_description'])

100%|██████████████████████████████████████████████████████| 18727/18727 [00:21<00:00, 858.48it/s]


In [194]:
response = requests.post(
    f"{SEARCH_INDEX}/_search",
    json={
        "size": 0,
        "track_total_hits": True
    }
)
response.json()

{'took': 33,
 'timed_out': False,
 'terminated_early': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 9295, 'relation': 'eq'},
  'max_score': None,
  'hits': []}}

In [195]:
def search_query(query='#$query##'):
    return {
      "query": {
        "multi_match": {
          "query": query,
          "fields": [f"name", "description"]
        }
      }
    }
    
def search(query):
    response = requests.post(
        f"{SEARCH_INDEX}/_search",
        json=search_query(query)
    )
    return response.json()

In [196]:
search('dinosaur')

{'took': 45,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 31, 'relation': 'eq'},
  'max_score': 13.017879,
  'hits': [{'_index': 'megacities',
    '_id': '1975317',
    '_score': 13.017879,
    '_source': {'name': 'Baby Dinosaur Balloon Set for Birthday Decor - 38 Inch, Pack of 4, 4D Dinosaur Foil Balloon | Kids Dinosaur Party Decorations | Dinosaur Balloons for Birthday Party | Dinosaur Birthday Party Supplies',
     'description': '<p><b>Are you a dinasaur fanatic?</b></p><p>Looking for a dinosaur theme party decorations for your kid</p> <p>We have got you covered with these beautiful and gigantic <b>Baby Dinosaur Foil Balloons </b> in two different style of dinosaur party balloons </p> <p>This dinosaur balloon kit is simply beautiful, gorgeous color to your dinosaur balloon birthday party as a backdrop for photos Booth.</p> <p>Configure it any way you wish; there are 1000s of ways to use your kit to build you

## Quepid

### init

In [197]:
!docker compose run quepid-api-quepid bin/rake db:migrate
!docker compose run quepid-api-quepid bin/rake db:seed
!docker compose run quepid-api-quepid bundle exec thor user:create -a admin@example.com "Admin User" supersecret

[+] create 2/2
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
 ✔ Container es-test                          Running                       0.0s
[+] start 2/2
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
 ✔ Container es-test                          Running                       0.0s
[+]  2/2
 ✔ Container judge-student-quepid-api-mysql-1 Running                       0.0s
 ✔ Container es-test                          Running                       0.0s
Container es-test Waiting 
Container judge-student-quepid-api-mysql-1 Waiting 
Container judge-student-quepid-api-mysql-1 Healthy 
Container es-test Healthy 
Container judge-student-quepid-api-quepid-run-c88aca5b8357 Creating 
Container judge-student-quepid-api-quepid-run-c88aca5b8357 Created 

!!! RubyLLM's legacy acts_as API is deprecated and will be removed in RubyLLM 2.0.0. Please consult the migration guide at https://rubyllm.com/upgrading-to-1-7/

WARN[0000] Found o

In [198]:
import re

out = !docker compose run quepid-api-quepid bundle exec thor user:add_api_key admin@example.com

found = re.search(r"[0-9a-f]{64}", "\n".join(out))
if not found:
    raise RuntimeError("no API key in output:\n" + "\n".join(out))

QUEPID_TOKEN = found.group()

In [199]:
QUEPID_TOKEN

'57cc03133eff508f8533a1aa09c894308da099a7ab36a2b6c08f7b9f6ebc37ea'

### !!

In [200]:
AUTH = {
    "Authorization": f"Bearer {QUEPID_TOKEN}"
}

In [201]:
team = requests.post(
    'http://localhost:8081/api/teams/', 
    headers = AUTH,
    json={
        "name": "Justice Department"
    }   
)
team = team.json()

In [202]:
team

{'id': 1,
 'name': 'Justice Department',
 'created_at': '2026-09-06T14:53:20.350Z',
 'updated_at': '2026-09-06T14:53:20.350Z'}

In [203]:
endpoint = requests.post(
    'http://localhost:8081/api/search_endpoints/', 
    headers = AUTH,
    json={
        "name": "Megacities",
        "endpoint_url": f"http://quepid-api-elasticsearch:9200/{SEARCH_INDEX}/_search",
        "search_engine": "es",
        "api_method": "POST",
        "proxy_requests": 1,   
    }   
)
endpoint = endpoint.json()

In [204]:
print(endpoint)

{'id': 1, 'name': 'Megacities', 'owner': 1, 'search_engine': 'es', 'endpoint_url': 'http://quepid-api-elasticsearch:9200/http://localhost:9200/megacities/_search', 'api_method': 'POST', 'custom_headers': None, 'archived': 0, 'created_at': '2026-09-06T14:53:25.293Z', 'updated_at': '2026-09-06T14:53:25.293Z', 'basic_auth_credential': None, 'mapper_code': None, 'proxy_requests': 1, 'options': None, 'requests_per_minute': None, 'test_query': None}


In [205]:
# list scorers
scorers = requests.get(
    'http://localhost:8081/api/scorers/', 
    headers = AUTH
)
print({s['id']: s['name'] for s in scorers.json()['items']})

{1: 'nDCG@10', 2: 'DCG@10', 3: 'CG@10', 4: 'P@10', 5: 'AP@10', 6: 'RR@10', 7: 'ERR@10'}


In [206]:
# Two books, one rater each, and that is the whole point of this layout.
#
# Quepid syncs a book's judgements down to its cases -- and to *every* case
# attached to that book: RatingsManager#sync_ratings_for_query_doc_pair does
# `@book.cases.each`, with no way to narrow it on this version. Worse, it
# blends every rater on a pair into one number via
# calculate_rating_from_judgements (top three, the agreed value if they agree,
# otherwise the minimum). So two cases hanging off one book would end up with
# identical, averaged ratings, and the human/AI comparison would disappear
# exactly when both sides had rated.
#
# One rater per book avoids it: with a single rater the blend is a no-op, and
# each case gets its own side of the experiment.
def make_book(name):
    r = requests.post(
        'http://localhost:8081/api/books/',
        headers = AUTH,
        json={
            "name": name,
            # ESCI's four labels in descending relevance, so a judgement can be
            # compared with esci_label directly. A book with no scale renders no
            # rating buttons at all.
            "scale": [0, 1, 2, 3],
            "scale_with_labels": {
                "0": "Irrelevant",
                "1": "Complement",
                "2": "Substitute",
                "3": "Exact",
            },
        }
    )
    r.raise_for_status()
    return r.json()


def make_case(name, book):
    r = requests.post(
        'http://localhost:8081/api/case/',
        headers = AUTH,
        json={
            "name": name,
            "scorer_id": 1,
            "book_id": book['id'],
            "search_endpoint_id": endpoint.get('id'),
            "search_query": json.dumps(search_query()),
        }
    )
    r.raise_for_status()
    return r.json()


In [207]:
# the humans' side.
# ESCI's labels go in as judgements, and no AI
# judge is ever attached to this book, so nothing overwrites them.
book_esci = make_book("Australo-Hungaria")
case_esci = make_case("Australo-Hungaria", book_esci)

# Neo-Chicago -- the machine's side. The same pairs, but no labels: every
# judgement in this book comes from the AI judge.
book_ai = make_book("Neo-Chicago")
case_ai = make_case("Neo-Chicago", book_ai)

# {c['name']: {"case": c['id'], "book": c['book_id']} for c in (case_esci, case_ai)}

In [208]:
book_esci, case_esci

({'scale': [0, 1, 2, 3],
  'scale_with_labels': {'0': 'Irrelevant',
   '1': 'Complement',
   '2': 'Substitute',
   '3': 'Exact'},
  'id': 1,
  'name': 'Australo-Hungaria',
  'created_at': '2026-09-06T14:53:40.944Z',
  'updated_at': '2026-09-06T14:53:40.944Z',
  'support_implicit_judgements': 0,
  'show_rank': 0,
  'owner_id': 1,
  'export_job': None,
  'import_job': None,
  'populate_job': None,
  'archived': 0,
  'scoring_guidelines': None},
 {'id': 1,
  'case_name': 'Australo-Hungaria',
  'last_try_number': 1,
  'owner': 1,
  'archived': 0,
  'scorer_id': 1,
  'created_at': '2026-09-06T14:53:40.976Z',
  'updated_at': '2026-09-06T14:53:40.976Z',
  'book_id': 1,
  'public': None,
  'options': None,
  'nightly': 1})

In [209]:
# A book's queries are the distinct query_texts of its query/doc pairs, and a
# pair must carry a doc -- so the ground truth goes in whole, one pair per df
# row. doc_id is example_id, matching what index_record used as the
# Elasticsearch _id, so the book lines up with what a search on myindex returns
# (and joins straight back onto df, esci_label included).
df_pairs = df[df['product_title'].notna() & df['product_description'].notna()]

pairs = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "document_fields": {
            "name": row['product_title'],
            "description": row['product_description'],
        },
        # Nothing in query_doc_pairs holds a foreign id, so the ESCI query_id
        # travels in options. Deliberately not esci_label: the judge reads the
        # document, and the label is recoverable from doc_id anyway.
        "query_options": {"esci_query_id": int(row['query_id'])},
    }
    for _, row in df_pairs.iterrows()
]
print(f"{df_pairs['query'].nunique()} queries, {len(pairs)} pairs")


# Batched: a pair is identified by (query_text, doc_id), so re-running this
# cell adds nothing and a batch that fails can simply be sent again.
def load_pairs(book):
    written = {"created": 0, "skipped": 0}
    for i in tqdm(range(0, len(pairs), 500), desc=book['name']):
        r = requests.post(
            f"http://localhost:8081/api/books/{book['id']}/query_doc_pairs/",
            headers = AUTH,
            json=pairs[i:i + 500]
        )
        r.raise_for_status()
        for k, v in r.json().items():
            written[k] += v
    return written


# Both books get the identical corpus. They differ only in who judges it.
{book['name']: load_pairs(book) for book in (book_esci, book_ai)}

945 queries, 9295 pairs


Neo-Chicago: 100%|████████████████████████████████████████████████| 19/19 [00:01<00:00, 16.81it/s]


{'Australo-Hungaria': {'created': 9295, 'skipped': 0},
 'Neo-Chicago': {'created': 9295, 'skipped': 0}}

In [210]:
# The labels, into Babilon only. A pair carries no rating: the rating is a
# *judgement*, one row per (rater, pair), so ESCI's ground truth goes in as
# judgements of ours. Mega-City One deliberately gets none of this -- its
# judgements are the AI judge's work, and that is what makes the two cases
# comparable rather than contaminated.
ESCI_RATING = {"I": 0, "C": 1, "S": 2, "E": 3}   # the book's scale, set above

labels = [
    {
        "query_text": row['query'],
        "doc_id": str(row['example_id']),
        "rating": ESCI_RATING[row['esci_label']],
        "explanation": f"ESCI ground truth: {row['esci_label']}",
    }
    for _, row in df_pairs.iterrows()
]

# Identity is (rater, pair), so re-running updates rather than duplicating --
# unlike the pairs above, which are skipped. Rows naming a pair the book does
# not hold come back as "unknown" instead of failing the batch, so a non-zero
# count there means labels went nowhere and is worth looking at.
written = {"created": 0, "updated": 0, "unchanged": 0, "unknown": 0}
for i in tqdm(range(0, len(labels), 500)):
    r = requests.post(
        f"http://localhost:8081/api/books/{book_esci['id']}/judgements/",
        headers = AUTH,
        json=labels[i:i + 500]
    )
    r.raise_for_status()
    for k, v in r.json().items():
        written[k] += v

written

100%|█████████████████████████████████████████████████████████████| 19/19 [00:01<00:00, 13.87it/s]


{'created': 9295, 'updated': 0, 'unchanged': 0, 'unknown': 0}

In [211]:
# Both cases now need their queries. Attaching a book to a case copies nothing:
# a book holds query/doc *pairs*, a case holds *queries*, and cases.book_id is
# the only thing joining the two tables. A case with no queries runs nothing
# against the search endpoint and scores nothing.
df_case_queries = df_pairs[['query_id', 'query']].drop_duplicates('query')


def load_case_queries(case):
    # Read first, so the cell is re-runnable: nothing de-duplicates case
    # queries server-side, and there is no bulk endpoint for them either --
    # unlike pairs and judgements, this is one POST per query.
    have = requests.get(
        f"http://localhost:8081/api/query/{case['id']}/?limit=100000",
        headers = AUTH
    )
    have.raise_for_status()
    have = {q['query_text'] for q in have.json()['items']}

    added = 0
    for _, row in tqdm(df_case_queries.iterrows(),
                       total=len(df_case_queries), desc=case['case_name']):
        if row['query'] in have:
            continue
        q = requests.post(
            f"http://localhost:8081/api/query/{case['id']}/",
            headers = AUTH,
            # The ESCI query_id rides along here too, so a case query joins back
            # to df and to the book's pairs on something other than the text.
            json={"query_text": row['query'],
                  "query_options": {"esci_query_id": int(row['query_id'])}}
        )
        q.raise_for_status()
        added += 1
    return {"added": added, "already there": len(have)}


{case['case_name']: load_case_queries(case) for case in (case_esci, case_ai)}

Neo-Chicago: 100%|█████████████████████████████████████████████| 945/945 [00:04<00:00, 231.74it/s]


{'Australo-Hungaria': {'added': 945, 'already there': 0},
 'Neo-Chicago': {'added': 945, 'already there': 0}}

### AI Judge

In [212]:
llm_key = getpass.getpass("LLM API key: ")

LLM API key:  ········


In [213]:
judge = requests.post(
    'http://localhost:8081/api/ai_judges/',
    headers = AUTH,
    json={
        "name": "Dredd",
        "llm_key": llm_key,
        "judge_options": {"llm_model": "gpt-4o"},
        # system_prompt omitted, which takes Quepid's default: rate 0-3, answer
        # in JSON. That is where the scale lives as far as the model is
        # concerned -- LlmService sends it the query and the document fields and
        # nothing else, so the book's scale_with_labels never reaches it.
    }
)
judge.raise_for_status()
judge = judge.json()
judge['id'], judge['name'], judge['judge_options']

(2,
 'Dredd',
 {'llm_provider': 'openai',
  'llm_service_url': 'https://api.openai.com',
  'llm_model': 'gpt-4o',
  'llm_timeout': 30,
  'llm_api_version': ''})

In [214]:
# Mega-City One only. Sharing a team is not enough to make a judge usable on a
# book -- run_judge_judy picks from book.ai_judges, which is this join -- and
# attaching it to Babilon as well is precisely what this layout exists to
# prevent: its judgements would be blended into Babilon's ratings alongside
# ESCI's, and both cases would end up saying the same thing.
attached = requests.post(
    f"http://localhost:8081/api/ai_judges/{judge['id']}/books/",
    headers = AUTH,
    json={"book_id": book_ai['id']}
)
attached.raise_for_status()

# Idempotent, so re-running is a no-op -- unlike the cell above, which has no
# natural key and would make a second judge.
[(b['id'], b['name']) for b in requests.get(
    f"http://localhost:8081/api/ai_judges/{judge['id']}/books/", headers = AUTH
).json()]

[(2, 'Neo-Chicago')]

In [215]:
# Nothing here starts the judging run: run_judge_judy is an HTML route that
# enqueues RunJudgeJudyJob, and neither Quepid's API nor this one exposes it.
# Open Mega-City One's book, pick the judge, and let it work through the pairs.
# When the run finishes it enqueues UpdateCaseJob, which fills case Mega-City
# One's ratings from those judgements by itself -- Babilon is untouched, having
# no judge and nothing to enqueue.
print(f"http://localhost:3000/books/{book_ai['id']}/judgement_stats")

http://localhost:3000/books/2/judgement_stats


## train student

In [ ]:
compare

In [ ]:
human labels - better teacher - dspy??